# 1. Bag of Words (BoW)

Bag of Words (BoW) throws away order. For each document, count how many times each vocabulary word appears. Vector length is the vocabulary size. Position i is the count of word i.



In [10]:
def build_vocab(docs):
    vocab = {}
    for doc in docs:
        for token in doc:
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

Rows are documents. Columns are vocabulary indices. Entry [i][j] is "how many times word j appears in document i." Doc 1 has cat twice because it did. Doc 0 has ran zero times because it did not.

In [11]:
def bag_of_words(docs, vocab):
    matrix = [[0] * len(vocab) for _ in docs]
    for i, doc in enumerate(docs):
        for token in doc:
            if token in vocab:
                matrix[i][vocab[token]] += 1
    return matrix

In [12]:
docs = [["cat", "sat", "on", "mat"], ["cat", "cat", "ran"]]
vocab = build_vocab(docs)

In [13]:
vocab

{'cat': 0, 'sat': 1, 'on': 2, 'mat': 3, 'ran': 4}

In [14]:
bag_of_words(docs, vocab)


[[1, 1, 1, 1, 0], [2, 0, 0, 0, 1]]

# 2. TF-IDF

TF-IDF reweights BoW. A word that appears in every document is uninformative, so scale it down. A word rare across the corpus but frequent in a single document is signal, so scale it up.

In [15]:
import math


def term_frequency(doc_bow, doc_length):
    return [c / doc_length if doc_length else 0 for c in doc_bow]


def document_frequency(bow_matrix):
    df = [0] * len(bow_matrix[0])
    for row in bow_matrix:
        for j, count in enumerate(row):
            if count > 0:
                df[j] += 1
    return df


def inverse_document_frequency(df, n_docs):
    return [math.log((n_docs + 1) / (d + 1)) + 1 for d in df]

In [16]:
def tfidf(bow_matrix):
    n_docs = len(bow_matrix)
    df = document_frequency(bow_matrix)
    idf = inverse_document_frequency(df, n_docs)
    out = []
    for row in bow_matrix:
        length = sum(row)
        tf = term_frequency(row, length)
        out.append([tf_j * idf_j for tf_j, idf_j in zip(tf, idf)])
    return out

In [17]:
docs = [
        ["the", "cat", "sat"],
        ["the", "dog", "sat"],
        ["the", "cat", "ran"],
        ["the", "dog", "ran"],
    ]
vocab = build_vocab(docs)
bow = bag_of_words(docs, vocab)
tfidf(bow)

[[0.3333333333333333, 0.5036085412553302, 0.5036085412553302, 0.0, 0.0],
 [0.3333333333333333, 0.0, 0.5036085412553302, 0.5036085412553302, 0.0],
 [0.3333333333333333, 0.5036085412553302, 0.0, 0.0, 0.5036085412553302],
 [0.3333333333333333, 0.0, 0.0, 0.5036085412553302, 0.5036085412553302]]

In [18]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = ["the cat sat on the mat", "the dog sat on the mat", "the cat ran"]

bow_vectorizer = CountVectorizer()
bow = bow_vectorizer.fit_transform(docs)
print(bow_vectorizer.get_feature_names_out())
print(bow.toarray())

tfidf_vectorizer = TfidfVectorizer()
tfidf = tfidf_vectorizer.fit_transform(docs)
print(tfidf.toarray().round(3))

['cat' 'dog' 'mat' 'on' 'ran' 'sat' 'the']
[[1 0 1 1 0 1 2]
 [0 1 1 1 0 1 2]
 [1 0 0 0 1 0 1]]
[[0.395 0.    0.395 0.395 0.    0.395 0.613]
 [0.    0.492 0.374 0.374 0.    0.374 0.581]
 [0.548 0.    0.    0.    0.72  0.    0.425]]
